# MOSAIC Phone Number Detector
**Detection-only.** Read-only against source data. Every finding requires steward review before any action.

| Cell | What it does | Procedure §§ |
|---|---|---|
| 1 - Overview | This document | — |
| 2 - Rule Reference | Full rule definitions, standardization options, decision workflow | All |
| 3 - Config | Parameters / widgets | — |
| 4 - Constants | Tags, severity, regex patterns, format taxonomy, standardization rules | Format, Normalization |
| 5 - Discovery | Table & column inventory from information_schema | — |
| 6 - Profiler | Sample-based stats: format distribution, mixed-format rate, invalid count | Format, Validation |
| 7 - AI Classifier | Confirm phone columns; classify format and country scope | Format, Country |
| 8 - Rule Engine | One function per rule group | All |
| 9 - Write Results | Append findings to results Delta table | — |
| 10 - Summary | Blocking findings first, ranked by table and column | Validation, Enforcement |

> **Hard constraint:** UPDATE, MERGE, DELETE, CTAS on source tables are never executed.
> **Scope:** All STRING columns screened for phone-like content by name or sample pattern.
> **AI use:** ai_query() confirms phone column identity, classifies persisted format (E.164, digits-only, display), country scope, and semantic role.


# MOSAIC Phone Number Detector — Rule Reference

**Detection-only.** No source data is modified.

---

## Format selection (§Format)
| Rule tag | What it detects | Required action |
|---|---|---|
| `MIXED_FORMAT` | Column contains phones in more than one format | Standardize to one documented format; blocks certification |
| `UNDOCUMENTED_FORMAT` | AI could not determine persisted format | Steward documents format before any Silver transform |
| `FORMAT_INCONSISTENT_WITH_TYPE` | Format inconsistent with declared purpose | Align format to use case; document in data dictionary |

## Country code (§Country)
| Rule tag | What it detects | Required action |
|---|---|---|
| `MISSING_COUNTRY_CODE` | International numbers stored without country code, no companion column | Add country code or split into country_code + national_number |
| `MISSING_COMPANION_COLUMN` | Digits-only Silver/Gold column with no companion country_code column | Add companion column or document domestic-only in dictionary |

## Normalization (§Normalization)
| Rule tag | What it detects | Required action |
|---|---|---|
| `NON_DIGIT_CHARS_PRESENT` | Non-digit chars in digits-only column | Strip in Silver ETL |
| `LEADING_ONE_PRESENT` | U.S. domestic column has leading 1 without dictionary rule | Remove leading 1 in Silver ETL |

## Validation (§Validation)
| Rule tag | What it detects | Required action |
|---|---|---|
| `INVALID_PHONE_LENGTH` | Digit count outside 7-15 range | NULL or quarantine |
| `HIGH_INVALID_RATE` | > threshold of non-null values fail length validation | Escalate; block certification |
| `PARSE_FAILURE_PRESENT` | Known garbage/placeholder phone values | NULL or quarantine |

## Privacy (§Privacy)
| Rule tag | What it detects | Required action |
|---|---|---|
| `PII_PHONE_RISK` | Phone column detected — verify PII controls | Document and enforce access controls and masking |

---

## Enforcement
Mixed phone formats in certified contact dimensions block certification until standardized.


In [0]:
import re, json, uuid
from datetime import datetime
from typing import Any, Dict, List, Tuple, Set
from collections import defaultdict
from pyspark.sql import functions as F, DataFrame

def _w(name: str, default) -> str:
    try:
        dbutils.widgets.text(name, str(default))
        return dbutils.widgets.get(name)
    except Exception:
        return str(default)

CFG: Dict[str, Any] = {
    # Source
    "catalog":      _w("catalog",      "data_classification_source"),
    "schema":       _w("schema",       "phone_number_detector"),
    "table_filter": _w("table_filter", ""),
    "skip_views":   _w("skip_views",   "true").strip().lower() == "true",

    # Layer
    "layer": _w("layer", "Unknown"),   # Bronze | Silver | Gold | Unknown

    # Default country for domestic validation (ISO 3166-1 alpha-2)
    # Used when no country code is present and no companion column exists.
    "default_country": _w("default_country", "US"),

    # Sampling
    "sample_size": int(_w("sample_size", 10_000)),
    "seed":        int(_w("seed",        42)),

    # Detection thresholds
    "mixed_format_threshold":  float(_w("mixed_format_threshold",  "2.0")),
    "invalid_rate_threshold":  float(_w("invalid_rate_threshold",  "5.0")),
    "phone_content_threshold": float(_w("phone_content_threshold", "50.0")),

    # AI
    "ai_model":  _w("ai_model",  "databricks-meta-llama-3-3-70b-instruct"),
    "enable_ai": _w("enable_ai", "true").strip().lower() == "true",

    # Results
    "out_catalog": _w("out_catalog", "data_classification_target"),
    "out_schema":  _w("out_schema",  "data_classification_output"),
    "out_table":   _w("out_table",   "phone_findings"),
}

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow().isoformat()

print(f"Run            : {RUN_ID}")
print(f"Scope          : {CFG['catalog']}.{CFG['schema']}")
print(f"Layer          : {CFG['layer']}")
print(f"Default country: {CFG['default_country']}")
print(f"AI             : {'on -> ' + CFG['ai_model'] if CFG['enable_ai'] else 'off (heuristic-only)'}")
print(f"Results        : {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")


In [0]:
# ---------------------------------------------------------------------------
# HOW TO ADD A NEW RULE:
#   1. Add tag to TAGS with procedure section reference
#   2. Add severity to SEVERITY
#   3. Add standardization options to STANDARDIZATION_RULES
#   4. Add detection logic in _check_phone() in Cell 6
# ---------------------------------------------------------------------------

PHONE_FORMATS = {"e164", "digits_only", "display", "mixed", "unknown"}

TAGS = {
    "MIXED_FORMAT":                    "§Format-selection",
    "UNDOCUMENTED_FORMAT":             "§Format-selection",
    "FORMAT_INCONSISTENT_WITH_TYPE":   "§Format-selection",
    "MISSING_COUNTRY_CODE":            "§Country-code",
    "MISSING_COMPANION_COLUMN":        "§Country-code",
    "NON_DIGIT_CHARS_PRESENT":         "§Normalization",
    "LEADING_ONE_PRESENT":             "§Normalization",
    "INVALID_PHONE_LENGTH":            "§Validation",
    "HIGH_INVALID_RATE":               "§Validation",
    "PARSE_FAILURE_PRESENT":           "§Validation",
    "PII_PHONE_RISK":                  "§Privacy",
}

SEVERITY: Dict[str, int] = {
    "MIXED_FORMAT":                  10,
    "HIGH_INVALID_RATE":             10,
    "INVALID_PHONE_LENGTH":           9,
    "PARSE_FAILURE_PRESENT":          8,
    "MISSING_COUNTRY_CODE":           7,
    "MISSING_COMPANION_COLUMN":       6,
    "NON_DIGIT_CHARS_PRESENT":        6,
    "LEADING_ONE_PRESENT":            5,
    "PII_PHONE_RISK":                 5,
    "UNDOCUMENTED_FORMAT":            4,
    "FORMAT_INCONSISTENT_WITH_TYPE":  3,
}

# Column name heuristics
RE_PHONE_NAME = re.compile(
    r"(phone|phone_number|phone_num|fax|fax_number|fax_num|"
    r"cell|cell_phone|mobile|mobile_number|telephone|tel|"
    r"contact_phone|work_phone|home_phone|alt_phone|"
    r"primary_phone|secondary_phone|emergency_phone|"
    r"office_phone|direct_phone|pager|pager_number)",
    re.I
)
RE_PHONE_EXCLUDE = re.compile(
    r"(_type$|_format$|_status$|_flag$|_ind$|_indicator$|"
    r"_count$|_code$|_notes$|_comment$)",
    re.I
)
RE_COUNTRY_CODE_NAME = re.compile(
    r"(country_code|country_cd|phone_country|intl_code|"
    r"calling_code|dial_code|region_code)",
    re.I
)

# Phone pattern regexes
RE_E164        = re.compile(r"^\+[1-9]\d{6,14}$")
RE_DIGITS_ONLY = re.compile(r"^\d{7,15}$")
RE_DISPLAY     = re.compile(r"^[\d\s\(\)\-\.\+x]{7,20}$")
RE_PHONE_LIKE  = re.compile(r"[\d]{7,}")

MIN_PHONE_DIGITS = 7
MAX_PHONE_DIGITS = 15

GARBAGE_PHONES = {
    "0000000000", "1111111111", "2222222222", "3333333333",
    "4444444444", "5555555555", "6666666666", "7777777777",
    "8888888888", "9999999999", "1234567890", "0123456789",
    "+10000000000", "+19999999999",
    "none", "null", "n/a", "na", "unknown", "tbd", "test",
}

STANDARDIZATION_RULES: Dict[str, list] = {

    "MIXED_FORMAT": [
        {"option_key": "standardize_to_e164",
         "label":      "Standardize all values to E.164 in Silver transform",
         "sql":        "-- SILVER: mosaic_gov.dq.phone_parse(col, '<country>').e164 AS col",
         "notes":      "E.164 is the preferred single-column international format (§Format). "
                       "Pin library version (phonenumbers==9.0.32) in pipeline docs. "
                       "Mixed formats in certified contact dimensions block certification (§Enforcement)."},
        {"option_key": "standardize_to_digits_only",
         "label":      "Standardize all values to digits-only national format in Silver",
         "sql":        "-- SILVER: REGEXP_REPLACE(col, '[^0-9]', '') AS col",
         "notes":      "Document chosen format in data dictionary (§Format). "
                       "Apply same format across all rows in the column (§Format)."},
        {"option_key": "document_format_and_remediate",
         "label":      "Document one format in data dictionary then remediate",
         "sql":        "-- Document format in data dictionary first, then apply transform.",
         "notes":      "One persisted format per column must be documented (§Format)."},
    ],

    "UNDOCUMENTED_FORMAT": [
        {"option_key": "steward_document_format",
         "label":      "Steward documents persisted format in data dictionary",
         "sql":        "-- No transform until format is documented (§Format).",
         "notes":      "Document one persisted format per column: E.164, digits-only national, "
                       "or documented display pattern (§Format)."},
    ],

    "FORMAT_INCONSISTENT_WITH_TYPE": [
        {"option_key": "align_format_to_use_case",
         "label":      "Align format to column use case and document in data dictionary",
         "sql":        "-- SILVER: apply format consistent with use case.",
         "notes":      "Document format choice in data dictionary (§Format)."},
    ],

    "MISSING_COUNTRY_CODE": [
        {"option_key": "add_country_code_prefix",
         "label":      "Add country code to produce E.164 or split into two columns",
         "sql":        "-- Option A: CONCAT('+1', REGEXP_REPLACE(col, '[^0-9]', '')) AS col  (US)\n"
                       "-- Option B (preferred): country_code INT, national_number STRING separately",
         "notes":      "Option B preferred for mixed domestic/international (§Country). "
                       "Validate national length (10 digits for US) after split."},
    ],

    "MISSING_COMPANION_COLUMN": [
        {"option_key": "add_companion_country_code_column",
         "label":      "Add companion country_code column or document domestic-only rule",
         "sql":        "-- Option B: ADD COLUMN country_code INT DEFAULT 1  (US domestic)\n"
                       "-- Or document in data dictionary that all values are domestic.",
         "notes":      "Option B (country_code + national_number) preferred for mixed sources (§Country)."},
    ],

    "NON_DIGIT_CHARS_PRESENT": [
        {"option_key": "strip_non_digits",
         "label":      "Strip non-digit characters in Silver transform",
         "sql":        "-- digits-only: REGEXP_REPLACE(col, '[^0-9]', '') AS col\n"
                       "-- E.164: REGEXP_REPLACE(col, '[^0-9+]', '') AS col  (keep leading +)",
         "notes":      "Strip non-digit characters during transform except leading + for E.164 (§Normalization)."},
    ],

    "LEADING_ONE_PRESENT": [
        {"option_key": "remove_leading_one",
         "label":      "Remove leading 1 from U.S. domestic digits-only values in Silver",
         "sql":        "-- SILVER: CASE WHEN col RLIKE '^1[2-9][0-9]{9}$' "
                       "THEN SUBSTRING(col, 2) ELSE col END AS col",
         "notes":      "Remove leading 1 from U.S. numbers only when dictionary defines "
                       "domestic storage rules (§Normalization)."},
    ],

    "INVALID_PHONE_LENGTH": [
        {"option_key": "null_invalid",
         "label":      "Set invalid-length phone values to NULL in Silver",
         "sql":        "-- SILVER: CASE WHEN LENGTH(REGEXP_REPLACE(col,'[^0-9]','')) "
                       "BETWEEN 7 AND 15 THEN col ELSE NULL END AS col",
         "notes":      "Invalid numbers must be NULL or quarantined (§Validation). "
                       "Do not persist partial garbage strings in certified layers."},
        {"option_key": "quarantine_invalid",
         "label":      "Route invalid-length rows to quarantine table",
         "sql":        "-- Route rows WHERE LENGTH(REGEXP_REPLACE(col,'[^0-9]','')) "
                       "NOT BETWEEN 7 AND 15 to quarantine.",
         "notes":      "Use when steward needs to review invalid values before nulling."},
    ],

    "HIGH_INVALID_RATE": [
        {"option_key": "quarantine_and_escalate",
         "label":      "Quarantine all invalid rows and escalate to steward",
         "sql":        "-- Route all invalid rows to quarantine. Block certification.",
         "notes":      "Mixed phone formats in certified contact dimensions block certification (§Enforcement)."},
    ],

    "PARSE_FAILURE_PRESENT": [
        {"option_key": "null_unparseable",
         "label":      "Set unparseable/garbage phone values to NULL in Silver",
         "sql":        "-- SILVER: CASE WHEN LOWER(TRIM(col)) IN (<garbage_list>) "
                       "THEN NULL ELSE col END AS col",
         "notes":      "Invalid numbers must be NULL or quarantined (§Validation). "
                       "Pin library version (phonenumbers==9.0.32) in pipeline docs."},
    ],

    "PII_PHONE_RISK": [
        {"option_key": "verify_pii_controls",
         "label":      "Verify PII access controls and masking policy are documented",
         "sql":        "-- No transform. Document PII controls in data dictionary.",
         "notes":      "Phone numbers are often PII (§Privacy). "
                       "Enforce PII masking in non-production environments (§Privacy)."},
    ],
}

print(f"Constants loaded: {len(TAGS)} tags | {len(STANDARDIZATION_RULES)} standardization entries")
print(f"Phone formats   : {sorted(PHONE_FORMATS)}")
print(f"Default country : {CFG['default_country']}")


In [0]:
# ---------------------------------------------------------------------------
# Reads Unity Catalog information_schema.
# Builds table_cols (STRING candidates) and table_all_cols (all column names,
# for companion country_code column detection).
# ---------------------------------------------------------------------------

cat, sch = CFG["catalog"], CFG["schema"]

def _esc(name: str) -> str:
    return name.replace("`", "``")

STRING_DTYPES_PFX = ("STRING", "VARCHAR", "CHAR")

view_clause = "AND table_type IN ('MANAGED', 'EXTERNAL')" if CFG["skip_views"] else ""
tables: List[str] = [
    r.table_name for r in spark.sql(f"""
        SELECT table_name FROM `{_esc(cat)}`.information_schema.tables
        WHERE  table_schema = '{_esc(sch)}' {view_clause}
        ORDER BY table_name
    """).collect()
]

if CFG["table_filter"].strip():
    pat = re.compile(CFG["table_filter"], re.I)
    tables = [t for t in tables if pat.search(t)]

if not tables:
    raise ValueError(f"No tables found in {cat}.{sch} -- check catalog/schema/table_filter.")

tbl_in = ", ".join(f"'{_esc(t)}'" for t in tables)

table_cols:     Dict[str, List[Tuple[str, str]]] = defaultdict(list)
table_all_cols: Dict[str, Set[str]]              = defaultdict(set)

for r in spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM   `{_esc(cat)}`.information_schema.columns
    WHERE  table_schema = '{_esc(sch)}' AND table_name IN ({tbl_in})
    ORDER BY table_name, ordinal_position
""").collect():
    table_all_cols[r.table_name].add(r.column_name.lower())
    dt = r.data_type.upper()
    if any(dt.startswith(p) for p in STRING_DTYPES_PFX):
        table_cols[r.table_name].append((r.column_name, dt))

total_string_cols = sum(len(v) for v in table_cols.values())
print(f"Scope        : {cat}.{sch}")
print(f"Tables       : {len(tables)}")
print(f"String cols  : {total_string_cols}")
print(f"Layer        : {CFG['layer']}")


In [0]:
# ---------------------------------------------------------------------------
# PROFILER:
# 1. screen_phone_col(): name heuristic + sample content (RE_PHONE_LIKE).
# 2. profile_table_batch(): ONE SQL per table for ALL aggregate stats:
#    null_count, e164_count, digits_count, display_count,
#    short_count, long_count, nondigit_count, lead1_count, garbage_count.
# 3. Sample invalids collected from cached sample DF.
# 4. Companion country_code column detected from table_all_cols.
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")

def get_sample(tbl: str) -> DataFrame:
    fq = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    n  = CFG["sample_size"]
    try:
        return spark.sql(f"SELECT * FROM {fq} TABLESAMPLE ({n} ROWS)")
    except Exception:
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {fq}").collect()[0]["n"]
        return (
            spark.table(f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`")
            .sample(withReplacement=False, fraction=min(1.0, n / max(1, total)), seed=CFG["seed"])
            .limit(n)
        )


def screen_phone_col(col: str, sample_df: DataFrame) -> Tuple[bool, float]:
    name_match = bool(RE_PHONE_NAME.search(col)) and not bool(RE_PHONE_EXCLUDE.search(col))
    try:
        rows = (
            sample_df
            .select(F.col(f"`{_esc(col)}`").alias("v"))
            .filter(F.col("v").isNotNull() & (F.trim(F.col("v")) != ""))
            .limit(2000).collect()
        )
        total   = len(rows)
        if total == 0:
            return name_match, 0.0
        matched = sum(1 for r in rows if RE_PHONE_LIKE.search(str(r["v"]).strip()))
        rate    = matched / total * 100
        is_cand = name_match or rate >= CFG["phone_content_threshold"]
        return is_cand, rate
    except Exception:
        return name_match, 0.0


def profile_table_batch(tbl: str, candidates: Dict[str, dict]) -> Dict[str, dict]:
    if not candidates:
        return candidates

    fq    = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    exprs = ["COUNT(*) AS __total__"]

    garbage_sql = ", ".join(f"'{g}'" for g in GARBAGE_PHONES)

    for col, info in candidates.items():
        cs   = f"`{_esc(col)}`"
        safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        v    = f"TRIM({cs})"

        exprs += [
            f"COUNT(*) - COUNT({cs}) AS `__null__{safe}`",
            # Use [0-9] instead of \d -- character classes are unambiguous across
            # all Databricks Runtime versions; \d inside a Python f-string becomes
            # a single \d in the SQL string which some DBR versions misinterpret.
            # E.164: starts with +, then 1-9, then 6-14 more digits (total 7-15 digits)
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND {v} RLIKE '^[+][1-9][0-9]{{6,14}}$' THEN 1 END) AS `__e164__{safe}`",
            # digits-only: 7-15 consecutive digits, nothing else
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND {v} RLIKE '^[0-9]{{7,15}}$' THEN 1 END) AS `__digits__{safe}`",
            # display: contains at least 7 digits but has formatting chars too
            # (not pure digits and not E.164) -- e.g. (202) 555-1234, 202-555-1234
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND NOT ({v} RLIKE '^[+][1-9][0-9]{{6,14}}$') "
            f"  AND NOT ({v} RLIKE '^[0-9]{{7,15}}$') "
            f"  AND LENGTH(REGEXP_REPLACE({v}, '[^0-9]', '')) >= 7 "
            f"  THEN 1 END) AS `__display__{safe}`",
            # short: extracted digits < 7 (too few to be a valid phone)
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND LENGTH(REGEXP_REPLACE({v}, '[^0-9]', '')) < 7 "
            f"  AND LENGTH(REGEXP_REPLACE({v}, '[^0-9]', '')) > 0 "
            f"  THEN 1 END) AS `__short__{safe}`",
            # long: extracted digits > 15 (too many for E.164)
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND LENGTH(REGEXP_REPLACE({v}, '[^0-9]', '')) > 15 "
            f"  THEN 1 END) AS `__long__{safe}`",
            # non-digit chars (after stripping leading + for E.164 tolerance)
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND REGEXP_REPLACE({v}, '^[+]', '') RLIKE '[^0-9]' "
            f"  THEN 1 END) AS `__nondigit__{safe}`",
            # leading 1: 11-digit digits-only US number with redundant country code prefix
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND {v} RLIKE '^1[2-9][0-9]{{9}}$' THEN 1 END) AS `__lead1__{safe}`",
            f"COUNT(CASE WHEN LOWER(TRIM({cs})) IN ({garbage_sql}) "
            f"  THEN 1 END) AS `__garbage__{safe}`",
        ]

    try:
        row = spark.sql(f"SELECT {', '.join(exprs)} FROM {fq}").collect()[0].asDict()
    except Exception as e:
        print(f"  [WARN] Batch profile failed for {tbl}: {e}")
        return candidates

    total = row["__total__"] or 0
    for col, info in candidates.items():
        safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        info.update({
            "total":          total,
            "null_count":     row.get(f"__null__{safe}", 0) or 0,
            "e164_count":     row.get(f"__e164__{safe}", 0) or 0,
            "digits_count":   row.get(f"__digits__{safe}", 0) or 0,
            "display_count":  row.get(f"__display__{safe}", 0) or 0,
            "short_count":    row.get(f"__short__{safe}", 0) or 0,
            "long_count":     row.get(f"__long__{safe}", 0) or 0,
            "nondigit_count": row.get(f"__nondigit__{safe}", 0) or 0,
            "lead1_count":    row.get(f"__lead1__{safe}", 0) or 0,
            "garbage_count":  row.get(f"__garbage__{safe}", 0) or 0,
        })

    for col, info in candidates.items():
        cs = f"`{_esc(col)}`"
        try:
            sample_vals = [
                str(r["v"]) for r in
                table_samples.get(tbl, spark.range(0))
                .select(F.col(cs).alias("v"))
                .filter(F.col("v").isNotNull())
                .limit(6).collect()
            ]
            invalid_samples = [
                str(r["v"]) for r in
                table_samples.get(tbl, spark.range(0))
                .select(F.col(cs).alias("v"))
                .filter(
                    F.col("v").isNotNull() & (F.trim(F.col("v")) != "") &
                    (F.length(F.regexp_replace(F.trim(F.col(cs)), "[^0-9]", "")) < 7)
                ).limit(5).collect()
            ]
        except Exception:
            sample_vals     = []
            invalid_samples = []
        info["sample_vals"]     = sample_vals
        info["invalid_samples"] = invalid_samples

    return candidates


# Run profiler
table_samples:    Dict[str, DataFrame]       = {}
phone_candidates: Dict[str, Dict[str, dict]] = defaultdict(dict)
total_screened    = 0
total_candidates  = 0

for tbl in tables:
    cols      = table_cols.get(tbl, [])
    sample_df = get_sample(tbl)
    table_samples[tbl] = sample_df

    for col, dtype in cols:
        total_screened += 1
        if RE_PHONE_EXCLUDE.search(col):
            continue
        is_cand, phone_rate = screen_phone_col(col, sample_df)
        if not is_cand:
            continue

        candidate_reason = (
            "name_heuristic" if RE_PHONE_NAME.search(col) else "content_rate"
        )
        all_cols_lower   = table_all_cols.get(tbl, set())
        has_companion_cc = any(RE_COUNTRY_CODE_NAME.search(c) for c in all_cols_lower)

        phone_candidates[tbl][col] = {
            "dtype":              dtype,
            "total":              0,
            "null_count":         0,
            "e164_count":         0,
            "digits_count":       0,
            "display_count":      0,
            "short_count":        0,
            "long_count":         0,
            "nondigit_count":     0,
            "lead1_count":        0,
            "garbage_count":      0,
            "phone_rate":         phone_rate,
            "candidate_reason":   candidate_reason,
            "has_companion_cc":   has_companion_cc,
            "sample_vals":        [],
            "invalid_samples":    [],
            "phone_format":       "unknown",
            "format_source":      "heuristic",
            "format_confidence":  "low",
            "country_scope":      "unknown",
            "semantic_role":      "unknown",
            "confirmed":          True,
        }
        total_candidates += 1

    if phone_candidates[tbl]:
        phone_candidates[tbl] = profile_table_batch(tbl, phone_candidates[tbl])



print(f"Profiler done.")
print(f"  Screened  : {total_screened} string columns across {len(tables)} table(s)")
print(f"  Candidates: {total_candidates} phone column(s)")
for tbl, cols in phone_candidates.items():
    print(f"  {tbl}: {len(cols)} candidate(s)")


In [0]:
# ---------------------------------------------------------------------------
# AI PHONE CLASSIFIER -- chunked ai_query() calls (BATCH_SIZE per call).
#
# Step 1: Column confirmation (is this actually a phone column?)
#   - name_heuristic columns and content_rate >= 50% survive AI rejection
#   - pure content_rate < 50% are dropped on AI rejection
# Step 2: Format + country scope + semantic role classification.
# Heuristic fallback when AI fails.
# ---------------------------------------------------------------------------

BATCH_SIZE = 20

def _ai_query(prompt: str) -> str:
    safe = prompt.replace("'", "''")
    raw  = spark.sql(
        f"SELECT ai_query('{CFG['ai_model']}', '{safe}', failOnError => false) AS r"
    ).collect()[0]["r"]
    if hasattr(raw, "errorStatus") and raw.errorStatus:
        raise ValueError(raw.errorStatus)
    return raw.result if hasattr(raw, "result") else str(raw)

def _chunked(items: list, size: int = BATCH_SIZE):
    for i in range(0, len(items), size):
        yield items[i:i + size]

def _heuristic_format(info: dict) -> str:
    e164    = info.get("e164_count", 0)
    digits  = info.get("digits_count", 0)
    display = info.get("display_count", 0)
    non_null = max(1, info.get("total", 1) - info.get("null_count", 0))
    dominant = max(e164, digits, display)
    if dominant == 0:
        return "unknown"
    if dominant / non_null > 0.85:
        if dominant == e164:    return "e164"
        if dominant == digits:  return "digits_only"
        if dominant == display: return "display"
    return "mixed"


def classify_table(tbl: str) -> None:
    cands = phone_candidates.get(tbl, {})
    if not cands:
        return

    if not CFG["enable_ai"]:
        for col, info in cands.items():
            info["phone_format"]      = _heuristic_format(info)
            info["format_source"]     = "heuristic"
            info["format_confidence"] = "medium"
        return

    # Step 1: Column confirmation
    for chunk_items in _chunked(list(cands.items())):
        payload = json.dumps([{
            "col":          col,
            "table":        tbl,
            "samples":      info["sample_vals"][:6],
            "phone_rate":   round(info["phone_rate"], 1),
            "reason":       info["candidate_reason"],
            "e164_count":   info["e164_count"],
            "digits_count": info["digits_count"],
            "display_count":info["display_count"],
        } for col, info in chunk_items], default=str)

        prompt = (
            "Strict data classifier. Reply ONLY with a JSON array -- no prose, no markdown. "
            f"Table: {tbl}. Determine if each STRING column stores phone numbers. "
            "A phone column holds telephone, fax, mobile, or pager numbers. "
            "Not a phone column: postal codes, zip codes, ID numbers, SSN, credit card numbers. "
            f"Columns: {payload}. "
            'Return: [{"col":"<n>","is_phone":true|false,"confidence":"high|medium|low"}]'
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                col = item.get("col", "")
                if col not in cands:
                    continue
                if not item.get("is_phone", False):
                    info = phone_candidates[tbl][col]
                    if info["candidate_reason"] == "name_heuristic" or info["phone_rate"] >= 50.0:
                        info["ai_flagged_not_phone"] = True
                    else:
                        info["confirmed"] = False
        except Exception as e:
            print(f"  [WARN] AI confirm chunk failed for {tbl}: {e}")
            for col, info in chunk_items:
                if info["candidate_reason"] != "name_heuristic" and info["phone_rate"] < 50.0:
                    info["confirmed"] = False

    for col in list(phone_candidates[tbl].keys()):
        if not phone_candidates[tbl][col].get("confirmed", True):
            del phone_candidates[tbl][col]

    if not phone_candidates[tbl]:
        return

    # Step 2: Format + scope + role classification
    confirmed = list(phone_candidates[tbl].keys())
    for chunk in _chunked(confirmed):
        payload = json.dumps([{
            "col":              col,
            "table":            tbl,
            "samples":          phone_candidates[tbl][col]["sample_vals"][:6],
            "e164_count":       phone_candidates[tbl][col]["e164_count"],
            "digits_count":     phone_candidates[tbl][col]["digits_count"],
            "display_count":    phone_candidates[tbl][col]["display_count"],
            "has_companion_cc": phone_candidates[tbl][col]["has_companion_cc"],
        } for col in chunk if col in phone_candidates[tbl]], default=str)

        prompt = (
            "Healthcare data governance expert. Reply ONLY with a JSON array -- no prose, no markdown. "
            f"Table: {tbl}. For each confirmed phone column classify: "
            "(1) phone_format: 'e164' (+<country><national>, e.g. +12125551234), "
            "'digits_only' (digits only, e.g. 2125551234), "
            "'display' (formatted with dashes/parens/dots, e.g. (212) 555-1234), "
            "'mixed' (multiple formats in same column), 'unknown'. "
            "(2) country_scope: 'us_domestic', 'international', 'mixed', 'unknown'. "
            "(3) semantic_role: 'contact_phone', 'fax', 'mobile', 'emergency', "
            "'work_phone', 'home_phone', 'other'. "
            "(4) confidence: high/medium/low. "
            f"Columns: {payload}. "
            'Return: [{"col":"<n>","phone_format":"<f>","country_scope":"<s>",'
            '"semantic_role":"<r>","confidence":"high|medium|low"}]'
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                col = item.get("col", "")
                if col not in phone_candidates[tbl]:
                    continue
                phone_candidates[tbl][col].update({
                    "phone_format":      item.get("phone_format", "unknown"),
                    "format_source":     "ai",
                    "format_confidence": item.get("confidence", "low"),
                    "country_scope":     item.get("country_scope", "unknown"),
                    "semantic_role":     item.get("semantic_role", "other"),
                })
        except Exception as e:
            print(f"  [WARN] AI format chunk failed for {tbl}: {e}")
            for col in chunk:
                if col in phone_candidates[tbl]:
                    phone_candidates[tbl][col]["phone_format"]      = _heuristic_format(phone_candidates[tbl][col])
                    phone_candidates[tbl][col]["format_source"]     = "heuristic_fallback"
                    phone_candidates[tbl][col]["format_confidence"] = "low"
                    phone_candidates[tbl][col]["ai_error"]          = str(e)


for tbl in list(phone_candidates.keys()):
    classify_table(tbl)
    remaining = len(phone_candidates[tbl])
    fmts = {}
    for info in phone_candidates[tbl].values():
        f = info.get("phone_format", "unknown")
        fmts[f] = fmts.get(f, 0) + 1
    fmt_str = ", ".join(f"{f}:{n}" for f, n in sorted(fmts.items()))
    print(f"  ok {tbl}: {remaining} phone column(s) [{fmt_str}]")

total_confirmed = sum(len(v) for v in phone_candidates.values())
print(f"\nAI classification done. {total_confirmed} confirmed phone column(s).")


In [0]:
# ---------------------------------------------------------------------------
# RULE ENGINE: all stats from pre-computed phone_candidates dict.
# No additional full-table SQL runs here.
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")


def _finding(tbl, col, info, tag, count, total, samples, action,
             hint=None, confidence="high",
             std_options=None, auto_decided_action=None) -> dict:
    pct     = round(count / total * 100, 4) if total else 0.0
    options = std_options if std_options is not None else STANDARDIZATION_RULES.get(tag, [])
    decided = auto_decided_action if auto_decided_action is not None else None
    return {
        "run_id":                   RUN_ID,
        "run_ts":                   RUN_TS,
        "catalog":                  cat,
        "schema":                   sch,
        "table_name":               tbl,
        "column_name":              col,
        "data_type":                info.get("dtype", "UNKNOWN"),
        "layer":                    CFG["layer"],
        "phone_format":             info.get("phone_format", "unknown"),
        "format_source":            info.get("format_source", "heuristic"),
        "format_confidence":        info.get("format_confidence", "low"),
        "country_scope":            info.get("country_scope", "unknown"),
        "semantic_role":            info.get("semantic_role", "other"),
        "has_companion_cc":         bool(info.get("has_companion_cc", False)),
        "rule_ref":                 TAGS.get(tag, "§?"),
        "classification":           tag,
        "affected_count":           int(count),
        "affected_pct":             float(pct),
        "total_rows":               int(total),
        "sample_values":            json.dumps(samples, default=str),
        "recommended_action":       action,
        "dictionary_strategy_hint": hint,
        "standardization_rule":     json.dumps(options, ensure_ascii=False),
        "confidence":               confidence,
        "needs_steward_review":     decided is None,
        "decided_action":           decided,
        "decided_by":               None,
    }


def _check_phone(tbl: str, col: str, info: dict) -> list:
    total      = info["total"]
    null_count = info["null_count"]
    non_null   = total - null_count
    fmt        = info["phone_format"]
    fmt_conf   = info["format_confidence"]
    scope      = info["country_scope"]
    findings   = []

    if non_null == 0:
        return findings

    e164    = info["e164_count"]
    digits  = info["digits_count"]
    display = info["display_count"]
    invalid = info["short_count"] + info["long_count"] + info["garbage_count"]

    # §Format: UNDOCUMENTED_FORMAT
    if fmt == "unknown":
        findings.append(_finding(tbl, col, info, "UNDOCUMENTED_FORMAT",
            0, total, info["sample_vals"][:5],
            "Phone format could not be determined. "
            "Document one persisted format per column in the data dictionary: "
            "E.164, digits-only national, or documented display pattern (§Format). "
            "No Silver transform should be applied until format is documented.",
            confidence="low",
        ))
        return findings

    # §Format: MIXED_FORMAT
    # Two paths:
    # (A) fmt="mixed": AI explicitly detected multiple formats. Use profiler
    #     counts for the message; if all are 0 (e.g. regex mismatch on old data),
    #     still fire -- the AI classification is the authoritative signal here.
    # (B) fmt=single: fire only when profiler counts confirm actual secondary-format
    #     contamination. If profiler format counts sum to 0 (regex didn't match),
    #     skip -- we have no empirical evidence of a second format, and the column
    #     is likely clean. Do NOT infer contamination from a regex miss.
    _fmt_total_counted = e164 + digits + display   # sum of all profiler format counts
    if fmt == "mixed":
        # AI confirmed multiple formats -- always fire regardless of profiler counts
        if _fmt_total_counted > 0:
            fmt_detail = f"e164: {e164}, digits_only: {digits}, display: {display}"
        else:
            fmt_detail = "format breakdown pending re-profile after regex fix"
        findings.append(_finding(tbl, col, info, "MIXED_FORMAT",
            non_null, total, info["sample_vals"][:5],
            f"Column contains phone numbers in multiple formats ({fmt_detail}). "
            "Mixed phone formats in certified contact dimensions block certification (§Enforcement). "
            "Standardize to one documented format across all rows (§Format).",
            confidence="high",
        ))
    elif _fmt_total_counted > 0:
        # Single-format column: check for secondary contamination using profiler counts.
        # Only fire when we have actual counted evidence of a second format.
        format_counts  = {"e164": e164, "digits_only": digits, "display": display}
        dominant_n     = format_counts.get(fmt, 0)
        non_dominant_n = _fmt_total_counted - dominant_n   # counted rows NOT in dominant format
        if non_null > 0 and non_dominant_n > 0 and non_dominant_n / non_null * 100 > CFG["mixed_format_threshold"]:
            findings.append(_finding(tbl, col, info, "MIXED_FORMAT",
                non_dominant_n, total, info["sample_vals"][:5],
                f"Column is primarily {fmt} format but "
                f"{non_dominant_n:,} value(s) ({non_dominant_n/non_null*100:.1f}%) "
                "use a different format. "
                "Apply the same format across all rows in that column (§Format). "
                "Mixed formats in certified contact dimensions block certification (§Enforcement).",
                confidence="medium",
            ))
    # If _fmt_total_counted == 0 and fmt != "mixed": profiler regex returned no
    # counts (all values fell through all three patterns). This should not happen
    # after the [0-9] regex fix, but if it does, we do not fire a false-positive
    # MIXED_FORMAT -- the AI classified a single format and we have no evidence
    # of contamination. The UNDOCUMENTED_FORMAT path would catch a genuine unknown.

    # §Country: MISSING_COUNTRY_CODE
    if fmt in ("digits_only", "display") and scope in ("international", "mixed", "unknown"):
        if not info["has_companion_cc"]:
            findings.append(_finding(tbl, col, info, "MISSING_COUNTRY_CODE",
                non_null, total, info["sample_vals"][:5],
                f"Column stores phone numbers without an embedded country code "
                f"(format: {fmt}, scope: {scope}) and no companion country_code column "
                "was detected in this table. "
                "Option B (preferred): store country_code and national_number separately (§Country). "
                "Option A: use E.164 to embed country code in one column.",
                confidence="medium",
            ))

    # §Country: MISSING_COMPANION_COLUMN (Silver/Gold only)
    if fmt == "digits_only" and not info["has_companion_cc"] and CFG["layer"] in ("Silver", "Gold"):
        findings.append(_finding(tbl, col, info, "MISSING_COMPANION_COLUMN",
            0, total, info["sample_vals"][:5],
            f"Digits-only phone column in {CFG['layer']} layer with no companion "
            "country_code column detected in this table. "
            "Option B (preferred for mixed domestic/international): "
            "store country_code and national_number separately (§Country). "
            "If domestic-only, document this in the data dictionary.",
            confidence="medium",
        ))

    # §Normalization: NON_DIGIT_CHARS_PRESENT
    if fmt == "digits_only" and info["nondigit_count"] > 0:
        findings.append(_finding(tbl, col, info, "NON_DIGIT_CHARS_PRESENT",
            info["nondigit_count"], total, info["sample_vals"][:5],
            f"{info['nondigit_count']:,} value(s) contain non-digit characters "
            "in a digits-only phone column. "
            "Strip non-digit characters during Silver transform (§Normalization).",
            confidence="high",
            auto_decided_action="strip_non_digits",
        ))

    # §Normalization: LEADING_ONE_PRESENT
    if fmt == "digits_only" and info["lead1_count"] > 0:
        if scope in ("us_domestic", "unknown") and CFG["default_country"] == "US":
            findings.append(_finding(tbl, col, info, "LEADING_ONE_PRESENT",
                info["lead1_count"], total, info["sample_vals"][:5],
                f"{info['lead1_count']:,} value(s) have a leading 1 (U.S. country code prefix) "
                "in a digits-only domestic column. "
                "Remove leading 1 when dictionary defines domestic storage rules (§Normalization).",
                confidence="medium",
            ))

    # §Validation: INVALID_PHONE_LENGTH
    if invalid > 0:
        findings.append(_finding(tbl, col, info, "INVALID_PHONE_LENGTH",
            invalid, total, info["invalid_samples"][:5],
            f"{invalid:,} value(s) have digit counts outside the valid range "
            f"({MIN_PHONE_DIGITS}-{MAX_PHONE_DIGITS} digits). "
            "Invalid numbers must be set to NULL or quarantined (§Validation). "
            "Do not persist partial garbage strings in certified layers.",
            confidence="high",
        ))

    # §Validation: HIGH_INVALID_RATE
    if non_null > 0:
        invalid_pct = invalid / non_null * 100
        if invalid_pct > CFG["invalid_rate_threshold"]:
            findings.append(_finding(tbl, col, info, "HIGH_INVALID_RATE",
                invalid, total, info["invalid_samples"][:5],
                f"{invalid_pct:.1f}% of non-null values fail digit-length validation. "
                "Mixed phone formats in certified contact dimensions block certification (§Enforcement). "
                "Escalate to steward; quarantine all invalid rows.",
                confidence="high",
                auto_decided_action="quarantine_and_escalate",
            ))

    # §Validation: PARSE_FAILURE_PRESENT (known garbage placeholders)
    if info["garbage_count"] > 0:
        findings.append(_finding(tbl, col, info, "PARSE_FAILURE_PRESENT",
            info["garbage_count"], total, info["sample_vals"][:5],
            f"{info['garbage_count']:,} value(s) are known garbage/placeholder phone numbers. "
            "Invalid numbers must be set to NULL or quarantined (§Validation). "
            "Do not persist placeholder strings in certified layers.",
            confidence="high",
            auto_decided_action="null_unparseable",
        ))

    # §Privacy: PII_PHONE_RISK (always fires on confirmed phone columns)
    findings.append(_finding(tbl, col, info, "PII_PHONE_RISK",
        non_null, total, [],
        "Phone numbers are often PII (§Privacy). "
        "Verify access controls, masking policy, and compliance obligations "
        "are documented and enforced for this column. "
        "Enforce PII masking in non-production environments (§Privacy).",
        confidence="high",
        auto_decided_action="verify_pii_controls",
    ))

    return findings


all_findings: List[dict] = []

for tbl, cols in phone_candidates.items():
    tbl_count = 0
    for col, info in cols.items():
        col_findings = _check_phone(tbl, col, info)
        all_findings.extend(col_findings)
        tbl_count += len(col_findings)
    print(f"  ok {tbl}: {tbl_count} finding(s)")

print(f"\nRule engine done. {len(all_findings)} total finding(s).")


In [0]:
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, DoubleType, BooleanType)

SCHEMA = StructType([
    StructField("run_id",                   StringType(),  True),
    StructField("run_ts",                   StringType(),  True),
    StructField("catalog",                  StringType(),  True),
    StructField("schema",                   StringType(),  True),
    StructField("table_name",               StringType(),  True),
    StructField("column_name",              StringType(),  True),
    StructField("data_type",                StringType(),  True),
    StructField("layer",                    StringType(),  True),
    StructField("phone_format",             StringType(),  True),
    StructField("format_source",            StringType(),  True),
    StructField("format_confidence",        StringType(),  True),
    StructField("country_scope",            StringType(),  True),
    StructField("semantic_role",            StringType(),  True),
    StructField("has_companion_cc",         BooleanType(), True),
    StructField("rule_ref",                 StringType(),  True),
    StructField("classification",           StringType(),  True),
    StructField("affected_count",           LongType(),    True),
    StructField("affected_pct",             DoubleType(),  True),
    StructField("total_rows",               LongType(),    True),
    StructField("sample_values",            StringType(),  True),
    StructField("recommended_action",       StringType(),  True),
    StructField("dictionary_strategy_hint", StringType(),  True),
    StructField("standardization_rule",     StringType(),  True),
    StructField("confidence",               StringType(),  True),
    StructField("needs_steward_review",     BooleanType(), True),
    StructField("decided_action",           StringType(),  True),
    StructField("decided_by",               StringType(),  True),
])

out_fq  = f"`{CFG['out_catalog']}`.`{CFG['out_schema']}`.`{CFG['out_table']}`"
out_tbl = f"{CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}"

findings_df = spark.createDataFrame(all_findings or [], schema=SCHEMA)

if all_findings:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CFG['out_catalog']}`.`{CFG['out_schema']}`")
    findings_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(out_tbl)
    print(f"ok  {len(all_findings):,} finding(s) written to {out_fq}")
else:
    print("No findings generated.")

print(f"Run ID: {RUN_ID}")


In [0]:
BLOCKING = {"MIXED_FORMAT", "HIGH_INVALID_RATE", "INVALID_PHONE_LENGTH", "PARSE_FAILURE_PRESENT"}

if not all_findings:
    print("No phone number findings. Run complete.")
else:
    fdf = findings_df

    block_df = fdf.filter(F.col("classification").isin(BLOCKING)).orderBy(F.col("affected_pct").desc())
    n_block  = block_df.count()
    print("=" * 70)
    print(f"  SECTION 1 -- BLOCKING FINDINGS (certification blockers): {n_block}")
    print("=" * 70)
    if n_block:
        display(block_df.select(
            "table_name","column_name","phone_format","country_scope",
            "classification","rule_ref","affected_count","affected_pct",
            "recommended_action","decided_action","decided_by"
        ))
    else:
        print("  None.")

    print("\n" + "-" * 70)
    print("SECTION 2 -- Findings by classification")
    print("-" * 70)
    display(
        fdf.groupBy("classification","rule_ref","phone_format")
           .agg(
               F.count("*").alias("findings"),
               F.countDistinct("table_name").alias("tables"),
               F.countDistinct("column_name").alias("columns"),
               F.sum("affected_count").alias("total_affected"),
           ).orderBy(F.col("findings").desc())
    )

    print("\n" + "-" * 70)
    print("SECTION 3 -- Findings by table")
    print("-" * 70)
    display(
        fdf.groupBy("table_name")
           .agg(
               F.count("*").alias("total_findings"),
               F.sum(F.when(F.col("classification").isin(BLOCKING), 1).otherwise(0)).alias("blocking"),
               F.countDistinct("column_name").alias("phone_columns"),
               F.collect_set("phone_format").alias("formats_present"),
               F.collect_set("country_scope").alias("scopes_present"),
           ).orderBy(F.col("blocking").desc(), F.col("total_findings").desc())
    )

    fmt_df = fdf.filter(F.col("classification").isin(
        "MIXED_FORMAT","UNDOCUMENTED_FORMAT","MISSING_COUNTRY_CODE","MISSING_COMPANION_COLUMN"
    ))
    n_fmt = fmt_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 4 -- Format and country code issues ({n_fmt})")
    print("  Mixed formats and missing country codes block certification")
    print("-" * 70)
    if n_fmt:
        display(fmt_df.select(
            "table_name","column_name","phone_format","country_scope",
            "has_companion_cc","classification","affected_count",
            "recommended_action","decided_action","decided_by"
        ).orderBy("classification","table_name","column_name"))
    else:
        print("  None.")

    pii_df = fdf.filter(F.col("classification") == "PII_PHONE_RISK")
    n_pii  = pii_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 5 -- PII phone exposure ({n_pii})")
    print("  Phone numbers are often PII -- verify access controls and masking policy")
    print("-" * 70)
    if n_pii:
        display(pii_df.select(
            "table_name","column_name","semantic_role",
            "affected_count","recommended_action","decided_action","decided_by"
        ).orderBy("table_name","column_name"))

    work_df = fdf.filter(F.col("decided_action").isNotNull())
    n_work  = work_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 6 -- Engineer work queue ({n_work} decided)")
    print("-" * 70)
    if n_work:
        display(work_df.select(
            "table_name","column_name","phone_format","classification",
            "decided_action","decided_by","standardization_rule"
        ).orderBy("table_name","column_name"))
    else:
        print("  No decisions recorded yet.")
        print(f"  Query: SELECT * FROM {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")
        print(f"  WHERE run_id = '{RUN_ID}' AND decided_action IS NULL")

    print("\n" + "=" * 70)
    print(f"  Run: {RUN_ID}")
    print(f"  Scope: {cat}.{sch}  |  Layer: {CFG['layer']}")
    print(f"  Phone columns confirmed: {total_confirmed}")
    print(f"  Findings: {len(all_findings):,}  |  Blocking: {n_block}  |  Format/country: {n_fmt}  |  PII: {n_pii}")
    print("=" * 70)
    print("\nDetection-only. No source data was modified.")
    print("Every finding requires data steward review before any action.")
